In [3]:
# Install LangChain, Groq, and Google Sheets libraries quietly
!pip install -q langchain langchain-groq gspread

import os
import gspread
from google.colab import userdata, auth
from google.auth import default
from langchain_groq import ChatGroq

# Load Groq API key from Colab Secrets ("Groq") and initialize the LLM
os.environ["GROQ_API_KEY"] = userdata.get("Groq")
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

# Authenticate Google account and connect to Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open existing "Sweet Crumbs Orders" sheet or create it if running for the first time
try:
    sh = gc.open("Sweet Crumbs Orders")
except gspread.SpreadsheetNotFound:
    sh = gc.create("Sweet Crumbs Orders")
    sh.sheet1.append_row(["name", "item", "qty", "date", "eggless", "total"])

ws = sh.sheet1
print("Sweet Crumbs Sheet ready:", sh.url)

Sweet Crumbs Sheet ready: https://docs.google.com/spreadsheets/d/1Nji6clh-WP5SzsNk09_gTtETlZMCjBkXIUwih7ykKug


In [4]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

# Menu prices and bakery operating hours
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {
    "monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}

# Pydantic schema to validate order fields before writing to Google Sheets
class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")

# Tool 1: Look up menu prices
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

# Tool 2: Check opening hours for a given day
@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

# Tool 3: Validate and append a confirmed cake order to Google Sheets
@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

In [5]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for Sweet Crumbs Home Bakery.
FACTS:  Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

# Create the tool-calling agent
agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # Keep full agent trace as conversation memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

# Test opening hours, pricing, and logging a valid order to the sheet
print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))
print(chat("I'm Ravi. Please book 2 eggless chocolate cakes for 24 Dec."))

Yes! We’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost ₹1,300 in total. Let me know your name, quantity, and pickup date, and I’ll log the order for you.
Agent error: Error code: 400 - {'error': {'message': "Tool call validation failed: tool call validation failed: attempted to call tool 'get_price<|channel|>analysis' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "get_price<|channel|>analysis", "arguments": {"item":"chocolate cake","qty":2}}'}}
Sorry, I'm a little busy. Please try again in a minute.


In [6]:
# Print the step-by-step execution trace (user -> tool call -> tool result -> final reply)
for m in history:
    m.pretty_print()

# Test edge cases: item not on menu, missing pickup date, and invalid quantity (0)
print("\n--- Testing Customer Edge Cases ---")
edge_cases = [
    "Do you have a chocolate donut?",
    "Hi, I'm Ananya. Book 1 plum cake please.",
    "I'm Karthik. Book 0 cupcakes for tomorrow."
]

for q in edge_cases:
    history = []  # Reset history for each independent test case
    print(f"Customer: {q}")
    print(f"Agent: {chat(q)}\n")

================================ Human Message =================================

Are you open on Sunday? How much are 2 chocolate cakes?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_3bb273bc-a92f-4e26-99cd-25ee61661934)
 Call ID: fc_3bb273bc-a92f-4e26-99cd-25ee61661934
  Args:
    day: sunday
================================= Tool Message =================================
Name: is_open

Open 10 am to 6 pm on sunday.
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_94be3d0f-9710-48db-a71e-bac15305f899)
 Call ID: fc_94be3d0f-9710-48db-a71e-bac15305f899
  Args:
    item: chocolate cake
    qty: 2
================================= Tool Message =================================
Name: get_price

2 x chocolate cake = Rs 1300
================================== Ai Message ==================================

Yes! We’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost

In [7]:
# Create or open the Google Sheet for ShareSphere student service bookings
try:
    biz_sh = gc.open("ShareSphere Bookings")
except gspread.SpreadsheetNotFound:
    biz_sh = gc.create("ShareSphere Bookings")
    biz_sh.sheet1.append_row(["student_name", "service", "tasks", "deadline", "urgent", "total"])

biz_ws = biz_sh.sheet1
print("ShareSphere Sheet ready:", biz_sh.url)

# Service rates in rupees and campus mentor availability
CATALOGUE = {"web dev": 800, "ui/ux": 500, "ppt": 250, "data analytics": 700}
MENTOR_SLOTS = {
    "monday": "4 pm to 8 pm", "tuesday": "4 pm to 8 pm", "wednesday": "4 pm to 8 pm",
    "thursday": "4 pm to 8 pm", "friday": "4 pm to 8 pm", "saturday": "10 am to 6 pm",
    "sunday": "closed"
}

# Pydantic model for a validated ShareSphere booking record
class ServiceBooking(BaseModel):
    """A student freelance service booking extracted from a chat message."""
    student_name: str = Field(description="Student's first name")
    service: Literal["web dev", "ui/ux", "ppt", "data analytics"] = Field(description="Requested service")
    tasks: int = Field(description="Number of tasks or deliverables", ge=1)
    deadline: str = Field(description="Requested delivery date or deadline")
    urgent: bool = Field(description="True if the student asked for urgent or express delivery")

# Business Tool 1: Calculate total cost for a service
@tool
def get_service_price(service: str, tasks: int = 1) -> str:
    """Get the total price in rupees for a number of tasks in a ShareSphere service category.

    Args:
        service: Service name, e.g. "web dev", "ui/ux", "ppt", or "data analytics"
        tasks: Number of tasks requested
    """
    rate = CATALOGUE.get(service.lower().strip())
    if rate is None:
        return f"'{service}' is not offered. Available services: {', '.join(CATALOGUE)}."
    return f"{tasks} x {service} = Rs {rate * tasks}"

# Business Tool 2: Check campus mentor support hours
@tool
def check_mentor_hours(day: str) -> str:
    """Check ShareSphere campus mentor support hours for a day of the week, e.g. 'saturday'."""
    timing = MENTOR_SLOTS.get(day.lower().strip())
    return f"Mentor support on {day}: {timing}." if timing else f"'{day}' is not a valid day."

# Business Tool 3: Validate and save a confirmed booking to the ShareSphere sheet
@tool
def log_booking(student_name: str, service: str, tasks: int, deadline: str, urgent: bool) -> str:
    """Save a confirmed student service booking to the ShareSphere Bookings sheet.
    Use only when student_name, service, tasks, and deadline are all provided."""
    try:
        booking = ServiceBooking(
            student_name=student_name,
            service=service.lower().strip(),
            tasks=tasks,
            deadline=deadline,
            urgent=urgent
        )
    except Exception as e:
        return f"Could not save booking: {e}. Ask the student to fix the missing or invalid detail."
    total = CATALOGUE[booking.service] * booking.tasks
    biz_ws.append_row([booking.student_name, booking.service, booking.tasks,
                       booking.deadline, "yes" if booking.urgent else "no", total])
    return f"Saved: {booking.tasks} x {booking.service} for {booking.student_name} due {booking.deadline}, Rs {total}."

ShareSphere Sheet ready: https://docs.google.com/spreadsheets/d/1xLNJV2XJGirIiYPDaT4Au4KvUg2l9eUD2uN-MtuIhbQ


In [8]:
# Bind the Pydantic ServiceBooking schema to the LLM for structured extraction
structured_llm = llm.with_structured_output(ServiceBooking)

messy_messages = [
    "hey bro im sharif need 2 ppt decks urgently by friday pls",
    "Hi this is Priya, can someone do 1 web dev landing page for me before 30 Nov? not urgent",
    "yo Rahul here, want 3 ui/ux screens asap!!",
    "Hello, Sneha speaking. Looking for 2 data analytics dashboards by next Monday, normal delivery.",
    "Need 4 ppt slides done by tomorrow evening, rush job!! - Vikram"
]

# Parse all 5 messy messages into validated ServiceBooking records
for idx, msg in enumerate(messy_messages, 1):
    record = structured_llm.invoke(msg)
    print(f"{idx}. Input:  {msg}")
    print(f"   Record: {record.model_dump()}\n")

1. Input:  hey bro im sharif need 2 ppt decks urgently by friday pls
   Record: {'student_name': 'Sharif', 'service': 'ppt', 'tasks': 2, 'deadline': 'Friday', 'urgent': True}

2. Input:  Hi this is Priya, can someone do 1 web dev landing page for me before 30 Nov? not urgent
   Record: {'student_name': 'Priya', 'service': 'web dev', 'tasks': 1, 'deadline': '30 Nov', 'urgent': False}

3. Input:  yo Rahul here, want 3 ui/ux screens asap!!
   Record: {'student_name': 'Rahul', 'service': 'ui/ux', 'tasks': 3, 'deadline': '2026-11-27', 'urgent': True}

4. Input:  Hello, Sneha speaking. Looking for 2 data analytics dashboards by next Monday, normal delivery.
   Record: {'student_name': 'Sneha', 'service': 'data analytics', 'tasks': 2, 'deadline': '2026-12-01', 'urgent': False}

5. Input:  Need 4 ppt slides done by tomorrow evening, rush job!! - Vikram
   Record: {'student_name': 'Vikram', 'service': 'ppt', 'tasks': 4, 'deadline': 'tomorrow evening', 'urgent': True}



In [9]:
BIZ_SYSTEM = """ROLE:   You are the friendly booking assistant for ShareSphere, a student freelancing platform.
FACTS:  Only verified college students can book tasks. Payouts happen via UPI within 24 hours.
RULES:  Always use get_service_price for rates and check_mentor_hours for timings; never guess.
        Before log_booking, ensure you have student_name, service, tasks, and deadline; ask if missing.
        After saving, confirm the booking details and total in rupees.
FORMAT: Under 60 words, friendly, English or Telugu matching the student."""

biz_agent = create_agent(
    model=llm,
    tools=[get_service_price, check_mentor_hours, log_booking],
    system_prompt=BIZ_SYSTEM
)

biz_history = []
def chat_biz(text: str) -> str:
    global biz_history
    result = biz_agent.invoke({"messages": biz_history + [{"role": "user", "content": text}]})
    biz_history = result["messages"]
    return biz_history[-1].content

# Test checking mentor hours, calculating rates, and saving a booking to Google Sheets
print(chat_biz("Are mentors available on Saturday? Also how much for 2 ui/ux tasks?"))
print(chat_biz("I'm Sharif. Please book 2 urgent ui/ux tasks for 5 Dec."))

Yes! Mentors are available on Saturday from 10 am to 6 pm. Two UI/UX tasks cost Rs 1,000. Let me know if you’d like to book!
✅ Booking confirmed!  
• Student: Sharif  
• Service: UI/UX (2 urgent tasks)  
• Deadline: 5 Dec  
• Total: Rs 1,000  

You’ll receive the payout via UPI within 24 hours. Let me know if you need anything else!


In [10]:
# Define a poorly named tool with a vague docstring to observe how tool selection degrades
@tool
def tool1(service: str, tasks: int = 1) -> str:
    """does stuff"""
    rate = CATALOGUE.get(service.lower().strip())
    return f"{tasks} x {service} = Rs {rate * tasks}" if rate else "Not available."

broken_agent = create_agent(
    model=llm,
    tools=[tool1, check_mentor_hours, log_booking],
    system_prompt=BIZ_SYSTEM
)

# Test whether the agent still picks tool1 when asked for a price
broken_res = broken_agent.invoke({"messages": [{"role": "user", "content": "How much do 3 ppt tasks cost?"}]})
print("Response with vague 'tool1':", broken_res["messages"][-1].content)

# Restore the clean, properly documented get_service_price tool
fixed_agent = create_agent(
    model=llm,
    tools=[get_service_price, check_mentor_hours, log_booking],
    system_prompt=BIZ_SYSTEM
)
print("Restored agent ready with clear tool names and docstrings.")

Response with vague 'tool1': Sure! 3 PPT tasks come to ₹750 in total. Let me know if you’d like to book them, and I’ll need your name, the deadline, and whether it’s urgent.
Restored agent ready with clear tool names and docstrings.


In [11]:
# Run a multi-tool question and capture the full execution trace
trace_res = biz_agent.invoke({
    "messages": [{"role": "user", "content": "Are you open on Sunday and what is the cost of 2 web dev tasks?"}]
})

# Print every step in the trace using pretty_print()
for msg in trace_res["messages"]:
    msg.pretty_print()

# Count how many times the LLM was invoked vs. how many tool calls executed
model_calls = sum(1 for m in trace_res["messages"] if m.type == "ai")
tool_calls = sum(1 for m in trace_res["messages"] if m.type == "tool")

print(f"\nSummary -> Model calls: {model_calls} | Tool calls: {tool_calls}")

================================ Human Message =================================

Are you open on Sunday and what is the cost of 2 web dev tasks?
================================== Ai Message ==================================
Tool Calls:
  check_mentor_hours (fc_fd9a8546-b776-4e24-89b4-d2e7f04a312a)
 Call ID: fc_fd9a8546-b776-4e24-89b4-d2e7f04a312a
  Args:
    day: sunday
================================= Tool Message =================================
Name: check_mentor_hours

Mentor support on sunday: closed.
================================== Ai Message ==================================
Tool Calls:
  get_service_price (fc_7ea79e07-4831-4ca2-af52-c8eb76a390bc)
 Call ID: fc_7ea79e07-4831-4ca2-af52-c8eb76a390bc
  Args:
    service: web dev
    tasks: 2
================================= Tool Message =================================
Name: get_service_price

2 x web dev = Rs 1600
================================== Ai Message ==================================

Hi! We’re closed on Sunday